# Deep learning model

## Setups

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
import os, math, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torch
import numpy as np
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import pytorch_lightning as pl
import torchmetrics
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor, EarlyStopping
import wandb

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/home-credit-default-risk/sample_submission.csv
/kaggle/input/competitions/home-credit-default-risk/bureau_balance.csv
/kaggle/input/competitions/home-credit-default-risk/POS_CASH_balance.csv
/kaggle/input/competitions/home-credit-default-risk/application_train.csv
/kaggle/input/competitions/home-credit-default-risk/HomeCredit_columns_description.csv
/kaggle/input/competitions/home-credit-default-risk/application_test.csv
/kaggle/input/competitions/home-credit-default-risk/previous_application.csv
/kaggle/input/competitions/home-credit-default-risk/credit_card_balance.csv
/kaggle/input/competitions/home-credit-default-risk/installments_payments.csv
/kaggle/input/competitions/home-credit-default-risk/bureau.csv


In [2]:
# ============================================================
# CELL 1: Install if needed
# ============================================================
!pip install wandb -q

# ============================================================
# CELL 2: Login to W&B
# ============================================================
import wandb
wandb.login()  # Will ask for API key
# Get your key from: https://wandb.ai/authorize

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ilukianets (ilukianets-kyiv-school-of-economics) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Data Split

In [3]:
# Reproducibility helpers
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Select only numeric columns (integers and floats)
df_train = pd.read_csv("/kaggle/input/competitions/home-credit-default-risk/application_train.csv")
df_test = pd.read_csv("/kaggle/input/competitions/home-credit-default-risk/application_test.csv")

df_train = df_train.select_dtypes(include=['number'])
df_test = df_test.select_dtypes(include=['number'])

print(f'Numeric Train shape: {df_train.shape}')
print(f'Numeric Test shape: {df_test.shape}')

from sklearn.model_selection import train_test_split

# Perform the 80/20 stratified split on the entire dataframe
df_train_split, df_val_split = train_test_split(
    df_train, 
    test_size=0.20, 
    stratify=df_train['TARGET'], 
    random_state=42
)

print(f"Train dataframe shape: {df_train_split.shape}")
print(f"Validation dataframe shape: {df_val_split.shape}")
print(f"Target rate in train: {df_train_split['TARGET'].mean():.4f}")
print(f"Target rate in val: {df_val_split['TARGET'].mean():.4f}")

df_train_split_median = df_train_split.median()
df_train_split = df_train_split.fillna(df_train_split_median)
df_val_split = df_val_split.fillna(df_train_split_median)
df_test = df_test.fillna(df_train_split_median)

cuda
Numeric Train shape: (307511, 106)
Numeric Test shape: (48744, 105)
Train dataframe shape: (246008, 106)
Validation dataframe shape: (61503, 106)
Target rate in train: 0.0807
Target rate in val: 0.0807


## Classes Dataset and DataCollator

In [4]:
class PandasDataset(Dataset):
    """Handles datasets with or without labels"""
    
    def __init__(self, df, feature_cols, label_col=None, transforms=None):
        """
        Args:
            df: pandas DataFrame
            feature_cols: list of feature column names
            label_col: target column name (None for test data)
            transforms: optional transforms
        """
        self.df = df.reset_index(drop=True)
        self.feature_cols = feature_cols
        self.label_col = label_col
        self.transforms = transforms
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        features = row[self.feature_cols].values.astype(np.float32)
        
        if self.transforms:
            features = self.transforms(features)
        
        # If there's a target column, include it
        if self.label_col is not None:
            label = row[self.label_col]
            return {
                "features": features,
                "label": label
            }
        # If no target (test data), return only features
        else:
            return {
                "features": features,
                "label": None
            }

In [5]:
class DataCollator:
    """Converts batches to tensors with normalization"""
    
    def __init__(self, feature_mean=None, feature_std=None, dtype=torch.float32):
        """
        Args:
            feature_mean: numpy array of feature means
            feature_std: numpy array of feature stds
            dtype: torch dtype (float32, bfloat16, float16, etc.)
        """
        self.feature_mean = feature_mean
        self.feature_std = feature_std
        self.dtype = dtype
    
    def __call__(self, batch):
        # Extract features
        features_list = [sample["features"] for sample in batch]
        features = np.stack(features_list, axis=0)
        
        # Normalize features
        if self.feature_mean is not None and self.feature_std is not None:
            features = (features - self.feature_mean) / (self.feature_std + 1e-8)
        
        # Convert to specified dtype
        features = features.astype(np.float32)
        
        result = {
            "features": torch.from_numpy(features).to(self.dtype)
        }
        
        # Only add labels if they exist
        if batch[0]["label"] is not None:
            labels_list = [sample["label"] for sample in batch]
            labels = np.array(labels_list, dtype=np.float32)
            result["labels"] = torch.from_numpy(labels).long()
        
        return result

## Custom layers

In [8]:
class MyLinearLayerFirstVersion(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias = nn.Parameter(torch.zeros(out_features))
    
        nn.init.orthogonal_(self.weight)          
    
    def forward(self, x):
        return x @ self.weight.T + self.bias


class MyLinearLayerReimplementation(nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features

        self.weight = nn.Parameter(torch.empty(out_features, in_features))

        if bias:
            self.bias = nn.Parameter(torch.empty(out_features))
        else:
            self.register_parameter("bias", None)
        self.reset_parameters()

    def reset_parameters(self):
        # Kaiming uniform initialization for weights
        nn.init.kaiming_uniform_(self.weight,a=5**0.5)

        # Uniform initialization for bias
        if self.bias is not None:
            bound = 1 / self.in_features**0.5
            nn.init.uniform_(self.bias, -bound, bound)

    def forward(self, x):
        return x @ self.weight.T + self.bias


# Like BatchNorm but WITHOUT normalizing first
# Just learns to scale and shift each feature
# output = gamma * input + beta

class LearnableScaleShift(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(in_features))
        self.beta = nn.Parameter(torch.zeros(in_features))
    
    def forward(self, x):
        return self.gamma * x + self.beta



class FeatureGatingLayer(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.gate = nn.Parameter(torch.empty(in_features))
        nn.init.normal_(self.gate, mean=0, 
                        std=(1.0 / in_features) ** 0.5)
    
    def forward(self, x):
        return x * torch.sigmoid(self.gate)


# Learns attention weights for each feature
# output = input * softmax(learnable_weights)
# softmax ensures weights sum to 1

class FeatureAttentionLayer(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.attention = nn.Parameter(torch.empty(in_features))
        nn.init.zeros_(self.attention)
    
    def forward(self, x):
        weights = torch.softmax(self.attention, dim=-1)
        return x * weights


# Reimplement BatchNorm from scratch
# Formula:
# x_norm = (x - mean) / sqrt(var + eps)
# output = gamma * x_norm + beta

class MyBatchNorm(nn.Module):
    def __init__(self, in_features, eps=1e-5, momentum=0.1):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(in_features))
        self.beta = nn.Parameter(torch.zeros(in_features))
        self.eps = eps
        self.momentum = momentum
        
        self.register_buffer('running_mean', torch.zeros(in_features))
        self.register_buffer('running_var', torch.ones(in_features))
    
    def forward(self, x):
        if self.training:

            mean = x.mean(dim=0)
            var = x.var(dim=0, unbiased=False)
            

            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * mean
            self.running_var = (1 - self.momentum) * self.running_var + self.momentum * var
        else:

            mean = self.running_mean
            var = self.running_var
        

        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * x_norm + self.beta

## Adam optimizer

At each step t, for each parameter p:

1. Get gradient:        g  = p.grad

2. Update 1st moment:   m  = β1 * m + (1 - β1) * g
3. Update 2nd moment:   v  = β2 * v + (1 - β2) * g²

4. Bias correction:     m̂  = m / (1 - β1^t)
                        v̂  = v / (1 - β2^t)

5. Update parameter:    p  = p - lr * m̂ / (√v̂ + ε)

In [9]:
class MyAdam(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        defaults = dict(lr=lr, betas=betas, eps=eps)
        super().__init__(params, defaults)
        
    
    @torch.no_grad()  # No gradient tracking during update
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        
        # Loop over parameter groups
        for group in self.param_groups:
            
            beta1, beta2 = group['betas']  # Unpack betas
            lr           = group['lr']  # Learning rate
            eps          = group['eps']  # Epsilon
            
            #  Loop over parameters in group
            for p in group['params']:
                
                #  Skip if no gradient
                if p.grad is None:
                    continue
                
                grad = p.grad
                
                # Get state for this parameter
                state = self.state[p]
                
                # Step 1: Initialize state on first step
                if len(state) == 0:
                    state['step'] = 0          # Start at 0
                    state['m'] = torch.zeros_like(p)  # 1st moment (zeros)
                    state['v'] = torch.zeros_like(p)  # 2nd moment (zeros)
                

                # Step 2: Increment step
                state['step'] += 1
                t = state['step']
                

                # Step 3: Update moments
                # m = β1 * m + (1 - β1) * grad
                state['m'] = beta1 * state['m'] + (1 - beta1) * grad
                
                # v = β2 * v + (1 - β2) * grad²
                state['v'] = beta2 * state['v'] + (1 - beta2) * grad ** 2  
                
                # Step 4: Bias correction
                # m_hat = m / (1 - β1^t)
                m_hat = state['m'] / (1 - beta1 ** t)
                
                # v_hat = v / (1 - β2^t)
                v_hat = state['v'] / (1 - beta2 ** t)
                

                # STEP 5: Update parameter
                # p = p - lr * m_hat / (√v_hat + ε)
                p.data -= lr * m_hat / (torch.sqrt(v_hat) + eps)
        
        return loss

In [10]:
# Quick sanity check
model = nn.Linear(10, 1)
optimizer = MyAdam(model.parameters(), lr=1e-3)
optimizer_ref = torch.optim.Adam(model.parameters(), lr=1e-3)

x = torch.randn(32, 10)
y = torch.randn(32, 1)

loss = nn.MSELoss()(model(x), y)
loss.backward()
optimizer.step()
optimizer.zero_grad()

print(f"✅ Loss: {loss.item():.4f}")
print(f"✅ MyAdam works!" if not torch.isnan(loss) else "❌ NaN detected")

✅ Loss: 1.2330
✅ MyAdam works!


## Basic Model

In [12]:
class BasicModel(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.layers = nn.Sequential(
            
            # BLOCK 1: Custom layers
            FeatureGatingLayer(in_features),
            MyLinearLayerReimplementation(in_features, in_features // 2),
            MyBatchNorm(in_features // 2),
            nn.LeakyReLU(0.1),
            nn.Dropout(p=0.2),
            
            # BLOCK 2: Gradual reduction
            MyLinearLayerReimplementation(in_features // 2, in_features // 4),
            MyBatchNorm(in_features // 4),
            nn.LeakyReLU(0.1),
            nn.Dropout(p=0.2),
            
            MyLinearLayerReimplementation(in_features // 4, in_features // 8),
            MyBatchNorm(in_features // 8),
            nn.LeakyReLU(0.1),
            nn.Dropout(p=0.2),
            
            MyLinearLayerReimplementation(in_features // 8, in_features // 16),
            MyBatchNorm(in_features // 16),
            nn.LeakyReLU(0.1),
            nn.Dropout(p=0.2),
            
            # OUTPUT
            nn.Linear(in_features // 16, 1)
        )
    
    def forward(self, x):
        return self.layers(x)

# Now works for any feature count!
model = BasicModel(in_features=len(feature_cols))
x = torch.randn(32, len(feature_cols))
out = model(x)
print(out.shape)
print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.3f}M")

torch.Size([32, 1])
Parameters: 0.008M


## PyTorch Lightning: Training pipeline and logging

In [13]:
class LitBasicModel(pl.LightningModule):
    def __init__(self, in_features, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.model = BasicModel(in_features)
        self.criterion = nn.BCEWithLogitsLoss(pos_weight=None)
        
        # ✅ AUROC needs probabilities not hard labels
        self.train_metric = torchmetrics.AUROC(task="binary")
        self.val_metric = torchmetrics.AUROC(task="binary")
    
    def forward(self, x):
        return self.model(x)
    
    def training_step(self, batch, batch_idx):
        x = batch["features"]
        y = batch["labels"].float()
        
        logits = self(x)
        loss = self.criterion(logits.squeeze(), y)
        
        # ✅ Pass probabilities to AUROC
        probs = torch.sigmoid(logits.squeeze())
        self.train_metric.update(probs, y.long())
        
        self.log("train_loss", loss,
                 on_step=True, on_epoch=True, prog_bar=True)
        return loss
    
    def on_train_epoch_end(self):
        metric = self.train_metric.compute()
        self.log("train_auroc", metric, prog_bar=True)
        self.train_metric.reset()
    
    def validation_step(self, batch, batch_idx):
        x = batch["features"]
        y = batch["labels"].float()
        
        logits = self(x)
        loss = self.criterion(logits.squeeze(), y)
        
        # ✅ Pass probabilities to AUROC
        probs = torch.sigmoid(logits.squeeze())
        self.val_metric.update(probs, y.long())
        
        self.log("val_loss", loss, on_epoch=True, prog_bar=True)
    
    def on_validation_epoch_end(self):
        metric = self.val_metric.compute()
        self.log("val_auroc", metric, prog_bar=True)
        self.val_metric.reset()
    
    def configure_optimizers(self):
        optimizer = MyAdam(self.parameters(), lr=self.hparams.lr)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=10
        )
        return {"optimizer": optimizer, "lr_scheduler": scheduler}
    
    def predict_step(self, batch, batch_idx):
        x = batch["features"]
        logits = self(x)
        return torch.sigmoid(logits.squeeze())

In [14]:
feature_cols = df_train_split.columns[2:]

class CustomDataModule(pl.LightningDataModule):
    def __init__(self, batch_size=1024, num_workers=None):
        super().__init__()
        self.batch_size = batch_size
        self.num_workers = num_workers or max(os.cpu_count()-1, 1)
        
        # HINT 1: Compute normalization stats from training data ONLY
        feature_mean = df_train_split[feature_cols].mean().values
        feature_std  = df_train_split[feature_cols].std().values
        
        # HINT 2: Your collator handles normalization
        self.collator = DataCollator(
            feature_mean=feature_mean,
            feature_std=feature_std,
            dtype=torch.float32
        )
    
    def setup(self, stage=None):
        # HINT 3: Create 3 datasets
        # df_train_split → has labels
        # df_val_split   → has labels  
        # df_test        → NO labels (label_col=None)
        self.train_ds = PandasDataset(
            df=df_train_split,
            feature_cols=feature_cols,
            label_col="TARGET"
        )
        self.val_ds = PandasDataset(
            df=df_val_split,
            feature_cols=feature_cols,
            label_col="TARGET"
        )
        self.test_ds = PandasDataset(
            df=df_test,
            feature_cols=feature_cols,
            label_col=None       # ✅ None for test!
        )
    
    def train_dataloader(self):
        return DataLoader(
            self.train_ds,
            batch_size=self.batch_size,
            shuffle=True,        # Shuffle for training?
            pin_memory=torch.cuda.is_available(),
            num_workers=self.num_workers,
            collate_fn=self.collator
        )
    
    def val_dataloader(self):
        return DataLoader(
            self.val_ds,
            batch_size=self.batch_size,
            shuffle=False,        # Shuffle for validation?
            pin_memory=torch.cuda.is_available(),
            num_workers=self.num_workers,
            collate_fn=self.collator
        )
    
    def predict_dataloader(self):
        # HINT 4: Used by trainer.predict() for test predictions
        return DataLoader(
            self.test_ds,
            batch_size=self.batch_size,
            shuffle=False,        # Shuffle for test?
            pin_memory=torch.cuda.is_available(),
            num_workers=self.num_workers,
            collate_fn=self.collator
        )

In [16]:
# W&B Logger
wandb_logger = WandbLogger(
    project="Deep learning assignment 1",
    name="BasicModel-run-1",
    log_model=True
)

# Callbacks
checkpoint_cb = ModelCheckpoint(
    monitor="val_auroc",
    mode="max",
    save_top_k=1,
    filename="model-{epoch:02d}-{val_auroc:.4f}"
)

lr_monitor = LearningRateMonitor(logging_interval="step")

early_stopping = EarlyStopping(
    monitor="val_auroc",
    patience=5,
    mode="max",
    verbose=True
)


trainer = pl.Trainer(
    max_epochs=10,
    logger=wandb_logger,
    callbacks=[checkpoint_cb, lr_monitor, early_stopping],
    accelerator="auto",
    devices=1,             
    log_every_n_steps=10,
    enable_progress_bar=True
)

# Model and datamodule
lit_model = LitBasicModel(
    in_features=len(feature_cols),
    lr=1e-3
)

datamodule = CustomDataModule(
    batch_size=1024,
    num_workers=max(os.cpu_count()-1, 1)
)

# Train!
trainer.fit(lit_model, datamodule=datamodule)

# Predictions
predictions = trainer.predict(
    lit_model,
    datamodule=datamodule,
    ckpt_path="best"
)

# Process predictions
all_preds = torch.cat(predictions).cpu().numpy()

submission = pd.DataFrame({
    "SK_ID_CURR": df_test["SK_ID_CURR"],
    "TARGET": all_preds
})

print(f"Submission shape: {submission.shape}")
print(f"\nPrediction stats:")
print(f"  Min:  {all_preds.min():.4f}")
print(f"  Max:  {all_preds.max():.4f}")
print(f"  Mean: {all_preds.mean():.4f}")

submission.to_csv("submission.csv", index=False)
print("\n✅ submission.csv saved!")

wandb.finish()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model        │ YourModel         │  7.6 K │ train │     0 │
│ 1 │ criterion    │ BCEWithLogitsLoss │      0 │ train │     0 │
│ 2 │ train_metric │ BinaryAUROC       │      0 │ train │     0 │
│ 3 │ val_metric   │ BinaryAUROC       │      0 │ train │     0 │
└───┴──────────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 7.6 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 7.6 K                                                                                                
Total estimated model params size (MB): 0.030                                                                      
Modules in train mode: 23                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

Metric val_auroc improved. New best score: 0.721
Metric val_auroc improved by 0.005 >= min_delta = 0.0. New best score: 0.725
Metric val_auroc improved by 0.002 >= min_delta = 0.0. New best score: 0.727
Metric val_auroc improved by 0.002 >= min_delta = 0.0. New best score: 0.729
Metric val_auroc improved by 0.002 >= min_delta = 0.0. New best score: 0.731
Metric val_auroc improved by 0.001 >= min_delta = 0.0. New best score: 0.732
Metric val_auroc improved by 0.000 >= min_delta = 0.0. New best score: 0.732
Metric val_auroc improved by 0.000 >= min_delta = 0.0. New best score: 0.732
`Trainer.fit` stopped: `max_epochs=10` reached.


Restoring states from the checkpoint path at ./Deep learning assignment 1/2t5mrldn/checkpoints/model-epoch=09-val_auroc=0.7323.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at ./Deep learning assignment 1/2t5mrldn/checkpoints/model-epoch=09-val_auroc=0.7323.ckpt


Output()

Submission shape: (48744, 2)

Prediction stats:
  Min:  0.0005
  Max:  0.4801
  Mean: 0.0909

✅ submission.csv saved!


epoch,▁▁▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▆▆▆▇▇██
lr-MyAdam,██████████▇▇▇▇▇▇▇▇▇▇▇▇▆▆▆▆▄▄▄▄▃▂▂▂▂▂▂▁▁▁
train_auroc,▁▃▃▅▆▇▇▇██
train_loss_epoch,█▄▂▁▁▁▁▁▁▁
train_loss_step,█▇▆▅▄▃▃▂▂▂▂▂▁▃▁▂▂▂▂▁▂▂▂▂▁▂▂▂▃▂▂▂▁▂▁▂▂▂▂▂
trainer/global_step,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▇▇▇█████
val_auroc,▃▁▅▆▇█████
val_loss,█▃▁▁▁▁▁▁▁▁
epoch,9
lr-MyAdam,2e-05
train_auroc,0.68291


In [18]:
!kaggle competitions submit -c home-credit-default-risk -f /kaggle/working/submission.csv  -m "Basic model only numerical features"

100%|████████████████████████████████████████| 873k/873k [00:00<00:00, 3.84MB/s]
Successfully submitted to Home Credit Default Risk